# Laboratorio estructural digital — Semana 2

**Edificio de Ingeniería, UANDES** · Grupo 7 — Pedro Castillo, Monserrat Cubillos, Eduardo Vergara

Este notebook corre el laboratorio completo de principio a fin:

```
planos DXF → geometría → áreas tributarias → OpenSees → verificaciones → JSON → Unity
```

**Regla de oro del proyecto:** OpenSees *calcula*, el JSON es la *fuente de verdad*,
Unity solo *muestra*. En este notebook no se calcula nada estructural que no venga
de OpenSees.

Correr con `Kernel → Restart & Run All`, o celda por celda para la defensa.

---
## 0. Preparación

Se agrega `src/` al path para poder importar los módulos del proyecto.

In [ ]:
import os
import sys


def raiz_del_proyecto(marca='src/modelo_edificio.py'):
    """
    Busca la raiz del repo subiendo desde el directorio actual.

    No se usa os.path.abspath('') a secas porque eso depende de DONDE se
    lanzo Jupyter: si se abre desde la carpeta de arriba, los imports y
    las rutas de datos fallan con un error poco claro.
    """
    d = os.path.abspath('')
    for _ in range(5):
        if os.path.exists(os.path.join(d, marca)):
            return d
        padre = os.path.dirname(d)
        if padre == d:
            break
        d = padre
    raise RuntimeError(
        'No encuentro la raiz del proyecto (falta ' + marca + '). '
        'Abre el notebook desde la carpeta P1L2_Grupo_7.')


RAIZ = raiz_del_proyecto()
os.chdir(RAIZ)                      # para que %run y las rutas relativas peguen
sys.path.insert(0, os.path.join(RAIZ, 'src'))

import openseespy.opensees as ops
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon as PolyPatch

import areas_tributarias as at
import modelo_edificio as M

print('OpenSees  ', ops.version())
print('matplotlib', matplotlib.__version__)
print('proyecto  ', RAIZ)

---
## 1. Geometría del edificio

Los ejes salen de la capa `RLE-EJES` del plano `2017_67-100.dxf` (cotas en cm,
convertidas a m). Los muros, de la capa `RLE-MURO` del mismo plano — extraídos
por script, no escritos a mano.

In [ ]:
print(f'Ejes X ({M.nX}): ', M.EJES_X)
print(f'Ejes Y ({M.nY}): ', M.EJES_Y)
print(f'Niveles ({M.nNiveles}):', M.NIVELES_Z)
print()
print(f'Planta        {M.EJES_X[-1]-M.EJES_X[0]:.2f} x {M.EJES_Y[-1]-M.EJES_Y[0]:.2f} m'
      f'  =  {M.AREA_PLANTA:.2f} m2')
print(f'Altura        {M.NIVELES_Z[-1]:.2f} m')
print(f'Nodos/piso    {M.NODOS_POR_PISO}')
print()
print(f"Hormigon      f'c = {M.FPC} MPa   Ec = {M.Ec/1000:.0f} MPa   gamma = {M.GAMMA} kN/m3")
print(f'J columna     {M.J_rectangular(0.50, 0.50):.6e} m4  (Saint-Venant)')

In [ ]:
muros = M.cargar_muros()
print(f'Muros extraidos del plano: {len(muros)}\n')
print(f"{'id':<5}{'x1':>8}{'y1':>8}{'x2':>8}{'y2':>8}{'largo':>8}{'esp':>7}")
print('-' * 52)
for m in muros[:10]:
    print(f"{m['id']:<5}{m['x1']:>8.2f}{m['y1']:>8.2f}{m['x2']:>8.2f}"
          f"{m['y2']:>8.2f}{m['largo']:>8.2f}{m['espesor']:>7.2f}")
if len(muros) > 10:
    print(f'... y {len(muros)-10} mas')

---
## 2. Áreas tributarias (el corazón del Lab)

La losa **no** se modela con elementos finitos. Su carga se transfiere a las vigas
trazando las **bisectrices a 45°** desde las esquinas de cada paño:

| caso | forma | área |
|---|---|---|
| `b ≤ a` (viga larga) | trapecio | `b(2a−b)/4` |
| `b > a` (viga corta) | triángulo | `a²/4` |

El módulo construye el **polígono real** de cada zona, así el área se puede
verificar por la fórmula del cordón — un camino independiente de la fórmula
analítica.

In [ ]:
trib = M.tributarias_por_viga()

suma = sum(r['area'] for r in trib.values())
interiores = sum(1 for r in trib.values() if len(r['poligonos']) == 2)
bordes = sum(1 for r in trib.values() if len(r['poligonos']) == 1)

print(f'Vigas con area tributaria : {len(trib)}  '
      f'({interiores} interiores, {bordes} de borde)')
print(f'Suma de areas tributarias : {suma:.6f} m2')
print(f'Area de planta            : {M.AREA_PLANTA:.6f} m2')
print(f'Error                     : {abs(suma - M.AREA_PLANTA):.3e} m2')

### 2.1 Verlo: el reparto dibujado

Cada zona coloreada es la losa que descarga sobre **una** viga. Se ve de
inmediato que los paños alargados reparten muy distinto a los cuadrados.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 8))

# matplotlib.colormaps en vez de plt.get_cmap: este ultimo esta
# deprecado y desaparece en versiones nuevas.
cmap = matplotlib.colormaps['tab20']
for k, (clave, reg) in enumerate(sorted(trib.items(), key=lambda t: str(t[0]))):
    for poli in reg['poligonos']:
        ax.add_patch(PolyPatch(poli, closed=True, facecolor=cmap(k % 20),
                               edgecolor='white', linewidth=0.8, alpha=0.75))

# Malla de ejes estructurales por encima
for x in M.EJES_X:
    ax.plot([x, x], [M.EJES_Y[0], M.EJES_Y[-1]], 'k-', lw=1.4, zorder=5)
for y in M.EJES_Y:
    ax.plot([M.EJES_X[0], M.EJES_X[-1]], [y, y], 'k-', lw=1.4, zorder=5)

# Muros
for m in muros:
    ax.plot([m['x1'], m['x2']], [m['y1'], m['y2']], color='crimson',
            lw=3.5, solid_capstyle='butt', zorder=6)

ax.set_aspect('equal')
ax.set_xlabel('X [m]'); ax.set_ylabel('Y [m]')
ax.set_title(f'Areas tributarias por bisectrices a 45 grados  |  '
             f'suma = {suma:.2f} m2 = area de planta\n'
             f'(negro: ejes estructurales, rojo: muros del plano)')
plt.tight_layout(); plt.show()

### 2.2 Por qué el reparto 50/50 estaba mal

La versión de la Semana 1 daba la mitad de la carga a las vigas X y la mitad a
las Y. En un paño cuadrado da lo mismo; en uno alargado, no.

In [ ]:
Lx, Ly = 10.00, 3.34     # un pano real de este edificio

larga_45 = at.area_tributaria_viga(Lx, Ly)
corta_45 = at.area_tributaria_viga(Ly, Lx)
cincuenta = Lx * Ly / 4.0

print(f'Pano {Lx} x {Ly} m\n')
print(f"{'':<18}{'viga larga':>12}{'viga corta':>12}")
print('-' * 42)
print(f"{'bisectrices 45':<18}{larga_45:>12.3f}{corta_45:>12.3f}")
print(f"{'reparto 50/50':<18}{cincuenta:>12.3f}{cincuenta:>12.3f}")
print()
print(f'El 50/50 DESCARGA la viga larga un {(1-cincuenta/larga_45)*100:.0f}% '
      f'y SOBRECARGA la corta un {(cincuenta/corta_45-1)*100:.0f}%.')
print()
print(f'Ambos conservan el area total: '
      f'{2*larga_45 + 2*corta_45:.3f} = {Lx*Ly:.3f} m2')
print('-> por eso el EQUILIBRIO GLOBAL NO DETECTA este error.')

---
## 3. Construir el modelo y resolver

`q_G = peso propio de losa + terminaciones = 25 × 0.25 + 1.5 = 7.75 kN/m²`

La carga se aplica **distribuida** sobre las vigas (`eleLoad -beamUniform`), con
`w = q·A_trib/L`. El peso propio de vigas, columnas y muros se agrega aparte.

In [ ]:
topo = M.construir_modelo()

print(f"Columnas   {len(topo['columnas']):>5}")
print(f"Vigas X    {len(topo['vigas_x']):>5}")
print(f"Vigas Y    {len(topo['vigas_y']):>5}")
print(f"Muros      {len(topo['muros']):>5}")
print(f"{'':-<20}")
print(f"Elementos  {topo['n_elementos']:>5}")
print(f"Nodos      {len(topo['coords']):>5}")
print(f"Apoyos     {len(topo['apoyos']):>5}   (empotrados, 6 GDL)")
print(f"Diafragmas {len(topo['diafragmas']):>5}   (rigidDiaphragm, uno por piso)")

In [ ]:
M.nuevo_patron()
carga_G = M.aplicar_carga_gravitacional(topo, M.Q_G, incluir_peso_propio=True)

ok = M.resolver()
print('Convergencia:', 'OK' if ok == 0 else 'FALLO')

sumRz = sum(ops.nodeReaction(n, 3) for n in topo['apoyos'])
uz_min = min(ops.nodeDisp(n, 3) for n in topo['coords'])

print(f'\nCarga aplicada G   {carga_G:>15.4f} kN')
print(f'Suma reacciones Rz {sumRz:>15.4f} kN')
print(f'Error equilibrio   {abs(carga_G-sumRz):>15.3e} kN')
print(f'\nUZ maximo          {uz_min*1000:>15.4f} mm')

---
## 4. Las 5 verificaciones del Lab

Se corre el script completo, el mismo que se usa fuera del notebook.

In [ ]:
%run verificar_lab2.py

### 4.1 El diafragma: por qué la verificación no es trivial

Un diafragma rígido **no** obliga a que todos los nodos tengan el mismo `ux`.
El piso se mueve como cuerpo rígido *en su plano* y, con carga excéntrica,
además **rota**:

$$u_{x,i} = u_{x,m} - r_z\,(y_i - y_m) \qquad u_{y,i} = u_{y,m} + r_z\,(x_i - x_m)$$

Por eso se verifica con carga **lateral y excéntrica**: bajo gravedad pura el
giro es ≈ 0 y la prueba se cumpliría sola sin probar nada.

In [ ]:
def giro_de_piso(con_muros):
    """Resuelve una carga lateral excentrica y devuelve el giro maximo."""
    t = M.construir_modelo(con_muros=con_muros)
    M.nuevo_patron()
    for lev in range(1, M.nNiveles):
        ops.load(M.id_nodo(lev, 0, 0), 10.0 * lev, 0.0, 0.0, 0.0, 0.0, 0.0)
    M.resolver()
    return max(abs(ops.nodeDisp(M.id_maestro(l), 6))
               for l in range(1, M.nNiveles))

sin_m = giro_de_piso(False)
con_m = giro_de_piso(True)

print(f'Giro maximo de piso SIN muros : {sin_m:.4e} rad')
print(f'Giro maximo de piso CON muros : {con_m:.4e} rad')
print(f'Reduccion                     : {(1-con_m/sin_m)*100:.0f}%')
print('\n-> los muros SI estan tomando torsion, no son solo dibujo.')

---
## 5. Exportar el modelo para Unity

El JSON lleva todo lo que Unity necesita **ya calculado**: nodos con sus
restricciones por GDL, elementos, **ejes locales**, diafragmas y las **áreas
tributarias como polígonos**.

Unity no deduce los ejes locales: los recibe. Deducirlos en C# sería duplicar la
convención de `geomTransf`, y esa copia terminaría divergiendo del modelo real.

In [ ]:
%run src/exportar_unity.py

---
## 6. Abrir el visor 3D

La primera vez hay que **compilar** la aplicación (unos minutos). Después
arranca en segundos y ya no hace falta abrir el editor de Unity.

> El modo *Play* del editor es interactivo y no se puede disparar desde fuera
> (`-batchmode` y Play son incompatibles). Por eso se compila una app
> standalone: eso sí se automatiza, y ejecutarla es un proceso normal.

In [ ]:
import lanzar_unity as U

print('Unity que pide el proyecto:', U.version_del_proyecto())
print('Unity encontrada          :', U.buscar_unity())
print('App compilada             :', os.path.exists(U.APP))

In [ ]:
# Compila la app. Solo hace falta la PRIMERA vez (o tras cambiar los .cs).
# Si ya existe, no la recompila.
U.construir_app()

In [ ]:
# Copia el ultimo modelo calculado y abre el visor.
# Esta es la celda de la demo: se puede correr las veces que sea.
U.abrir_visor()

**Controles del visor**

| Acción | Cómo |
|---|---|
| Orbitar | arrastrar con botón izquierdo |
| Panear | arrastrar con botón derecho |
| Zoom | rueda |
| Encuadrar todo | tecla `F` |
| Inspeccionar una barra | click sin arrastrar |

En el panel se prenden las capas: **apoyos, diafragmas, ejes locales, IDs,
áreas tributarias**, y hay filtro por piso.

Al seleccionar una barra contesta: `elementTag`, nodos, **restricciones por
GDL**, **ejes locales**, **área tributaria** y los **kN de losa** que le llegan,
con el chequeo `w·L = q·A` en vivo.

---

*Para trabajar en el visor (no para mostrarlo), `U.abrir_editor()` abre el
proyecto en Unity; ahí hay que apretar Play a mano.*